# 00 - Silver Layer: Data Cleaning & Structural Refactoring

## Objetivo Ejecutivo
Este notebook orquestra la transformación de la capa Bronze (`data.csv`) hacia una arquitectura Silver optimizada. Los objetivos críticos son:
1. **Depuración de Fugas (Data Leakage):** Eliminación de variables de reclamaciones (claims) y sumatorias redundantes.
2. **Refactorización de Categorías:** Reversión de variables One-Hot (Especialidad, Estado, Edad) a formato denso para optimizar memoria.
3. **Integridad Longitudinal:** Validación del pulso de 86 semanas por médico.
4. **Split Estructural:** Definición de folds de validación mediante `GroupKFold` para prevenir el *Temporal Data Leakage*.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold
import logging
import re

# Runtime Logging Configuration
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

## 1. Data Ingestion & Feature Pruning
Cargamos el dataset y aplicamos una poda agresiva de columnas que introducen ruido, colinealidad o fuga de datos (claims de Medicare y sumatorias pre-calculadas).

In [ ]:
DATA_PATH = "data.csv"

try:
    logging.info("Commencing high-volume data ingestion...")
    df = pd.read_csv(DATA_PATH, engine="pyarrow")
except FileNotFoundError:
    logging.warning("Source data.csv not found. Operating on dummy architecture for structural validation.")
    # Generamos un dataset de prueba con la estructura exacta detallada para poder ejecutar la lógica
    df = pd.DataFrame({
        'NUEVO_ID': np.repeat(np.arange(1, 1001), 86),
        'WEEK_ID': np.tile(np.arange(1, 87), 1000),
        'SPEC_GE': np.random.choice([0, 1], 86000), 'SPEC_IM': np.random.choice([0, 1], 86000),
        'STATE_NY': np.random.choice([0, 1], 86000), 'STATE_CA': np.random.choice([0, 1], 86000),
        '(1940, 1960]': np.random.choice([0, 1], 86000), '(1960, 1980]': np.random.choice([0, 1], 86000),
        'UC_TRX': np.random.rand(86000), 'UC_NRX': np.random.rand(86000),
        'RTE': np.random.rand(86000), 'SAMPLES': np.random.rand(86000),
        'N_CLM_1': np.random.rand(86000), 'METRIC_R4_16SUM': np.random.rand(86000),
        'TIME_GIDX': np.random.rand(86000), 'YEAR': 2024,
        'ATSEG': np.nan
    })
    # Etiquetamos solo a una subpoblación
    df.loc[df['NUEVO_ID'] <= 50, 'ATSEG'] = np.random.choice([1, 2, 3], 4300)

# Identifying garbage columns to prune based on strict logical groups
cols_to_drop = [
    col for col in df.columns 
    if col.startswith('N_CLM') 
    or col.endswith('_R4_16SUM') 
    or col.endswith('_R4_29SUM') 
    or col.endswith('_GIDX') 
    or col in ['YEAR', 'QTR', 'YEAR_QTR']
]

# Execute aggressive pruning
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')
logging.info(f"Pruned {len(cols_to_drop)} leakage/redundant columns. Residual columns: {df.shape[1]}")

## 2. Categorical Reversion (One-Hot to Dense)
Revertimos las columnas dummies a variables categóricas únicas. Esto no solo limpia el schema, sino que reduce significativamente el uso de memoria RAM al tratar con 1.8M de registros.

In [ ]:
def reverse_dummies_vectorized(dataframe, prefix, new_col_name):
    """
    Reverts one-hot encoded columns back to a single categorical column.
    Drops the original dummy columns to free up memory.
    """
    dummy_cols = [c for c in dataframe.columns if c.startswith(prefix)]
    if not dummy_cols: return dataframe
    
    # idxmax returns the name of the column with the value 1. We strip the prefix.
    dataframe[new_col_name] = dataframe[dummy_cols].idxmax(axis=1).str.replace(prefix, "", regex=False)
    dataframe.drop(columns=dummy_cols, inplace=True)
    return dataframe

logging.info("Executing categorical reversion for Specialty, State, and Age domains...")

# 1. Specialty Reversion
df = reverse_dummies_vectorized(df, "SPEC_", "SPECIALTY")

# 2. State Reversion
df = reverse_dummies_vectorized(df, "STATE_", "STATE")

# 3. Age Range specific reversion (targeting regex pattern like (1940, 1960])
age_cols = [c for c in df.columns if re.match(r'\(\d{4},\s?\d{4}\]', c)]
if age_cols:
    df['AGE_RANGE'] = df[age_cols].idxmax(axis=1)
    df.drop(columns=age_cols, inplace=True)

print("Categorical Reversion complete.")
if 'SPECIALTY' in df.columns:
    print(df[['SPECIALTY', 'STATE', 'AGE_RANGE']].head())

## 3. Label Flagging & Type Casting
Definimos la frontera entre datos etiquetados (Supervisados) y no etiquetados (Inferencia), y optimizamos los tipos numéricos.

In [ ]:
logging.info("Optimizing memory and broadcasting label status...")

# Numerical Downcasting for Prescription and Marketing Metrics
float_cols = df.select_dtypes(include=['float64']).columns
if len(float_cols) > 0:
    df[float_cols] = df[float_cols].astype('float32')
    
int_cols = df.select_dtypes(include=['int64']).columns
for col in int_cols:
    # Safely downcast integers
    df[col] = pd.to_numeric(df[col], downcast='integer')

# Label Propagation (is_labeled)
# Find HCPs that have at least one non-null value in the target column
labeled_hcps = df.loc[df['ATSEG'].notna(), 'NUEVO_ID'].unique()
df['IS_LABELED'] = df['NUEVO_ID'].isin(labeled_hcps)

print(f"Target Discovery: {len(labeled_hcps)} labeled HCPs identified out of {df['NUEVO_ID'].nunique()} total.")

## 4. Longitudinal Validation (CRÍTICO)
Aseguramos que la estructura temporal sea coherente. Cada HCP debe presentar una secuencia cronológica sin saltos.

In [ ]:
logging.info("Validating temporal continuity...")

# Strict Sorting by HCP and Time Sequence
df.sort_values(by=['NUEVO_ID', 'WEEK_ID'], ascending=[True, True], inplace=True)
df.reset_index(drop=True, inplace=True)

# Validate Sequence Lengths
hcp_counts = df.groupby('NUEVO_ID')['WEEK_ID'].count()
print("Sequence Length Distribution Statistics:")
print(hcp_counts.describe())

expected_seq_len = 86
integrity_check = (hcp_counts == expected_seq_len).all()
if not integrity_check:
    logging.warning(f"Structural Variance Detected: Some HCPs do not possess exactly {expected_seq_len} weeks.")
else:
    logging.info(f"Temporal Topology Validated: All sequences are {expected_seq_len} weeks long.")

## 5. Structural Split (GroupKFold Prevention)
Dividimos el dataset en 5 partes para validación cruzada. Usamos el `NUEVO_ID` como ancla para asegurar que un médico se trate como una unidad indivisible en entrenamiento vs prueba.

In [ ]:
logging.info("Instantiating GroupKFold manifold...")

# Set default validation fold to -1 for unlabeled inference data
df['VALIDATION_FOLD'] = -1

# We compute cross-validation folds ONLY over labeled data
labeled_mask = df['IS_LABELED'] == True
labeled_subset = df[labeled_mask].drop_duplicates('NUEVO_ID')[['NUEVO_ID']]

gkf = GroupKFold(n_splits=5)
splits = gkf.split(labeled_subset, groups=labeled_subset['NUEVO_ID'])

fold_map = {}
for fold_idx, (train_idx, val_idx) in enumerate(splits):
    val_hcps = labeled_subset.iloc[val_idx]['NUEVO_ID'].values
    for hcp in val_hcps:
        fold_map[hcp] = fold_idx

# Propagate fold mapping back to main dataframe safely
df.loc[labeled_mask, 'VALIDATION_FOLD'] = df.loc[labeled_mask, 'NUEVO_ID'].map(fold_map)

print("Validation Fold Distribution (Row Counts by Fold):")
print(df['VALIDATION_FOLD'].value_counts())

## 6. Silver Layer Export
Finalizamos la fase de limpieza exportando a Parquet, un formato columnar óptimo para datasets de alta densidad.

In [ ]:
logging.info("Executing programmatic sorting assertions...")

# Double-check temporal sorting using pandas shift()
df['PREV_NUEVO_ID'] = df['NUEVO_ID'].shift(1)
df['PREV_WEEK_ID'] = df['WEEK_ID'].shift(1)

same_hcp_mask = (df['NUEVO_ID'] == df['PREV_NUEVO_ID'])
assert (df.loc[same_hcp_mask, 'WEEK_ID'] > df.loc[same_hcp_mask, 'PREV_WEEK_ID']).all(), "Temporal Sorting Corrupted!"

df.drop(columns=['PREV_NUEVO_ID', 'PREV_WEEK_ID'], inplace=True)

OUTPUT_PATH = "silver_layer_longitudinal.parquet"
logging.info(f"Committing Silver Layer to disk: {OUTPUT_PATH}")

# Export preserving longitudinal structure without aggregations
df.to_parquet(OUTPUT_PATH, engine="pyarrow", index=False)

logging.info("Pipeline Step 00 Completed Successfully.")